## Gradient Boosting Implementation

In [ ]:
# importing libraries and data
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
# node class
class Node:
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction

# variance reduction
def variance_reduction(X_column, y, threshold):
    left_mask = X_column <= threshold
    right_mask = X_column > threshold
    y_left, y_right = y[left_mask], y[right_mask]
    if len(y_left) == 0 or len(y_right) == 0:
        return 0
    n = len(y)
    weighted_variance = (len(y_left)/n) * np.var(y_left) + \
                        (len(y_right)/n) * np.var(y_right)
    return np.var(y) - weighted_variance

# best split
def best_split_regression(X, y, n_features=None):
    best_gain = 0
    best_feature = None
    best_threshold = None
    n_total_features = X.shape[1]
    if n_features is None:
        n_features = n_total_features
    feature_indices = np.random.choice(n_total_features, n_features, replace=False)
    for feature in feature_indices:
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            gain = variance_reduction(X[:, feature], y, threshold)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold
    return best_feature, best_threshold

# regression tree builder
def build_tree_regression(X, y, max_depth=10, depth=0):
    if len(np.unique(y)) == 1:
        return Node(prediction=y[0])
    if depth >= max_depth:
        return Node(prediction=np.mean(y))
    if len(y) < 2:
        return Node(prediction=np.mean(y))
    feature, threshold = best_split_regression(X, y)
    if feature is None:
        return Node(prediction=np.mean(y))
    left_mask = X[:, feature] <= threshold
    right_mask = X[:, feature] > threshold
    left = build_tree_regression(X[left_mask], y[left_mask], max_depth, depth+1)
    right = build_tree_regression(X[right_mask], y[right_mask], max_depth, depth+1)
    return Node(feature=feature, threshold=threshold, left=left, right=right)

# predict functions
def predict_one(node, x):
    if node.prediction is not None:
        return node.prediction
    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict(node, X):
    return np.array([predict_one(node, x) for x in X])

# gradient boosting
class GradientBoostingFromScratch:
    def __init__(self, n_trees=50, max_depth=3, learning_rate=0.1):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.lr = learning_rate
        self.trees = []
        self.initial_prediction = None

    def fit(self, X, y):
        self.initial_prediction = np.mean(y)
        current_prediction = np.full(len(y), self.initial_prediction)
        for _ in range(self.n_trees):
            residuals = y - current_prediction
            tree = build_tree_regression(X, residuals, max_depth=self.max_depth)
            self.trees.append(tree)
            current_prediction += self.lr * predict(tree, X)

    def predict(self, X):
        pred = np.full(X.shape[0], self.initial_prediction)
        for tree in self.trees:
            pred += self.lr * predict(tree, X)
        return pred

# training
X, y = make_regression(n_samples=300, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# single tree baseline
single_tree = build_tree_regression(X_train, y_train, max_depth=3)
single_pred = predict(single_tree, X_test)
single_mse = mean_squared_error(y_test, single_pred)
print(f"Single tree MSE: {single_mse:.2f}")

# gradient boosting
print("\nMSE vs number of trees:")
for n in [1, 5, 10, 25, 50]:
    gb = GradientBoostingFromScratch(n_trees=n, max_depth=3, learning_rate=0.1)
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    print(f"n_trees={n:3d}  →  MSE: {mse:.2f}")

Single tree MSE: 2665.39

MSE vs number of trees:
n_trees=  1  →  MSE: 8763.16
n_trees=  5  →  MSE: 4741.16
n_trees= 10  →  MSE: 2866.99
n_trees= 25  →  MSE: 1286.64
n_trees= 50  →  MSE: 902.69


In [3]:
# 500 trees
for n in [50, 100, 200, 500]:
    gb = GradientBoostingFromScratch(n_trees=n, max_depth=3, learning_rate=0.1)
    gb.fit(X_train, y_train)
    pred = gb.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    print(f"n_trees={n:3d}  →  MSE: {mse:.2f}")

n_trees= 50  →  MSE: 901.62
n_trees=100  →  MSE: 777.60
n_trees=200  →  MSE: 751.43
n_trees=500  →  MSE: 750.89
